<a href="https://colab.research.google.com/github/charan327/RandomtreesvsDecisiontree/blob/main/Copy_of_Clustering_fitting_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
This is the template file for the clustering and fitting assignment.
You will be expected to complete all the sections and
make this a fully working, documented file.
You should NOT change any function, file or variable names,
 if they are given to you here.
Make use of the functions presented in the lectures
and ensure your code is PEP-8 compliant, including docstrings.
Fitting should be done with only 1 target variable and 1 feature variable,
likewise, clustering should be done with only 2 variables.
"""
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as ss
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler


In [ ]:
def plot_relational_plot(df):
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.scatterplot(
        data=df,
        x="work_hours_per_week",
        y="burnout_score",
        hue="gender",
        alpha=0.7,
        ax=ax
    )
    ax.set_title("Work Hours per Week vs Burnout Score")
    ax.set_xlabel("Work Hours per Week")
    ax.set_ylabel("Burnout Score")
    plt.tight_layout()
    plt.savefig("relational_plot.png")
    plt.close(fig)
    return

In [ ]:
def plot_categorical_plot(df):

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.barplot(
        data=df,
        x="burnout_level",
        y="stress_level",
        estimator=np.mean,
        errorbar=None,
        ax=ax
    )
    ax.set_title("Average Stress Level by Burnout Level")
    ax.set_xlabel("Burnout Level")
    ax.set_ylabel("Average Stress Level")
    plt.tight_layout()
    plt.savefig("categorical_plot.png")
    plt.close(fig)
    return

In [ ]:

def plot_statistical_plot(df):

    fig, ax = plt.subplots(figsize=(8, 6))
    corr_cols = [
        "work_hours_per_week",
        "sleep_hours",
        "stress_level",
        "anxiety_score",
        "burnout_score"
    ]
    corr = df[corr_cols].corr()
    sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", ax=ax)
    ax.set_title("Correlation Heatmap")
    plt.tight_layout()
    plt.savefig("statistical_plot.png")
    plt.close(fig)
    return

In [ ]:
def statistical_analysis(df, col: str):
    data = df[col].dropna()
    mean = np.mean(data)
    stddev = np.std(data, ddof=1)
    skew = ss.skew(data, bias=False)
    excess_kurtosis = ss.kurtosis(data, bias=False)
    return mean, stddev, skew, excess_kurtosis


In [ ]:
def preprocessing(df):
    print("First five rows:")
    print(df.head())

    print("\nLast five rows:")
    print(df.tail())

    print("\nDescriptive statistics:")
    print(df.describe(include="all"))

    df = df.drop_duplicates()
    df = df.dropna()

    numeric_df = df.select_dtypes(include=[np.number])
    print("\nCorrelation matrix:")
    print(numeric_df.corr())

    return df

In [ ]:
def writing(moments, col):
    print(f"For the attribute {col}:")
    print(f"Mean = {moments[0]:.2f}, "
          f"Standard Deviation = {moments[1]:.2f}, "
          f"Skewness = {moments[2]:.2f}, and "
          f"Excess Kurtosis = {moments[3]:.2f}.")

    if moments[2] > 0.5:
        skew_text = "right skewed"
    elif moments[2] < -0.5:
        skew_text = "left skewed"
    else:
        skew_text = "not strongly skewed"

    if moments[3] > 0:
        kurtosis_text = "leptokurtic"
    elif moments[3] < 0:
        kurtosis_text = "platykurtic"
    else:
        kurtosis_text = "mesokurtic"

    print(f"The data was {skew_text} and {kurtosis_text}.")
    return

In [ ]:
def perform_clustering(df, col1, col2):


    cluster_df = df[[col1, col2]].dropna()

    scaler = StandardScaler()
    data = scaler.fit_transform(cluster_df)

    inertias = []
    silhouette_scores = []
    k_values = range(2, 10)

    for k in k_values:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels_k = kmeans.fit_predict(data)
        inertias.append(kmeans.inertia_)
        silhouette_scores.append(silhouette_score(data, labels_k))

    best_k = list(k_values)[int(np.argmax(silhouette_scores))]

    def plot_elbow_method():

        fig, ax = plt.subplots(figsize=(8, 6))
        ax.plot(list(k_values), inertias, marker="o")
        ax.set_title("Elbow Method")
        ax.set_xlabel("Number of Clusters")
        ax.set_ylabel("Inertia")
        plt.tight_layout()
        plt.savefig("elbow_plot.png")
        plt.close(fig)
        return

    def one_silhouette_inertia():

        best_score = max(silhouette_scores)
        best_inertia = inertias[int(np.argmax(silhouette_scores))]
        return best_score, best_inertia

    _score, _inertia = one_silhouette_inertia()
    print(f"Best silhouette score: {_score:.4f}")
    print(f"Corresponding inertia: {_inertia:.4f}")
    print(f"Chosen number of clusters: {best_k}")

    plot_elbow_method()

    final_kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
    labels = final_kmeans.fit_predict(data)

    centres = final_kmeans.cluster_centers_
    xkmeans = centres[:, 0]
    ykmeans = centres[:, 1]
    cenlabels = np.arange(len(centres))

    return labels, data, xkmeans, ykmeans, cenlabels

In [ ]:
def plot_clustered_data(labels, data, xkmeans, ykmeans, centre_labels):

    fig, ax = plt.subplots(figsize=(8, 6))
    scatter = ax.scatter(
        data[:, 0],
        data[:, 1],
        c=labels,
        cmap="viridis",
        alpha=0.7
    )
    ax.scatter(
        xkmeans,
        ykmeans,
        c="red",
        marker="X",
        s=200,
        label="Centres"
    )

    for i, label in enumerate(centre_labels):
        ax.annotate(
            f"C{label}",
            (xkmeans[i], ykmeans[i]),
            textcoords="offset points",
            xytext=(5, 5)
        )

    ax.set_title("K-Means Clustering")
    ax.set_xlabel("Scaled Feature 1")
    ax.set_ylabel("Scaled Feature 2")
    ax.legend()
    plt.tight_layout()
    plt.savefig("clustering.png")
    plt.close(fig)
    return

In [ ]:
def perform_fitting(df, col1, col2):

    data = df[[col1, col2]].dropna()

    x_data = data[col1].values.reshape(-1, 1)
    y_data = data[col2].values

    model = LinearRegression()
    model.fit(x_data, y_data)

    x = np.linspace(data[col1].min(), data[col1].max(), 100)
    y = model.predict(x.reshape(-1, 1))

    r_squared = model.score(x_data, y_data)
    print(f"Linear regression R^2 score: {r_squared:.4f}")
    print(f"Intercept: {model.intercept_:.4f}")
    print(f"Slope: {model.coef_[0]:.4f}")

    return data, x, y

In [ ]:
def plot_fitted_data(data, x, y):

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(
        data.iloc[:, 0],
        data.iloc[:, 1],
        alpha=0.6,
        label="Observed data"
    )
    ax.plot(x, y, color="red", linewidth=2, label="Fitted line")
    ax.set_title("Linear Regression Fitting")
    ax.set_xlabel(data.columns[0])
    ax.set_ylabel(data.columns[1])
    ax.legend()
    plt.tight_layout()
    plt.savefig("fitting.png")
    plt.close(fig)
    return

In [ ]:
def main():

    df = pd.read_csv("data.csv")
    df = preprocessing(df)

    col = "burnout_score"

    plot_relational_plot(df)
    plot_statistical_plot(df)
    plot_categorical_plot(df)

    moments = statistical_analysis(df, col)
    writing(moments, col)

    clustering_results = perform_clustering(
        df,
        "stress_level",
        "burnout_score"
    )
    plot_clustered_data(*clustering_results)

    fitting_results = perform_fitting(
        df,
        "work_hours_per_week",
        "burnout_score"
    )
    plot_fitted_data(*fitting_results)
    return


if __name__ == "__main__":
    main()

First five rows:
   age  gender            job_role  experience_years company_size work_mode  \
0   50  Female   Backend Developer               7.8        Large    Hybrid   
1   36    Male  Frontend Developer               1.8     Mid-size    Remote   
2   29    Male              DevOps               2.5          MNC    Hybrid   
3   42  Female   Backend Developer               1.5     Mid-size    Hybrid   
4   40  Female  Frontend Developer               3.4        Large    Remote   

   work_hours_per_week  overtime_hours  meetings_per_day  deadlines_missed  \
0                 45.0             0.0               5.0                 0   
1                 56.0             4.0               6.0                 0   
2                 43.0             2.0               6.0                 3   
3                 57.0             9.0               4.0                 1   
4                 49.0             0.0               3.0                 4   

   ...  screen_time_hours  caffeine_int